In [ ]:
import pandas as pd
import numpy as np
import matplotlib

In [ ]:
cropdata = pd.read_csv(r"../dataset/crop_production.csv")
cropdata.head()

In [ ]:
raindata = pd.read_csv(r"../dataset/rainfall in india 1901-2015.csv")
raindata.head()

In [ ]:
pricedata = pd.read_csv(r"../dataset/Price_Agriculture_commodities_Week.csv")
pricedata.head()

In [ ]:
print(cropdata.shape)
print(raindata.shape)
print(pricedata.shape)

In [ ]:
cropdata.head()

In [ ]:
raindata.head()

In [ ]:
pricedata.head()

In [ ]:
cropdata.columns

In [ ]:
raindata.columns

In [ ]:
pricedata.columns

In [ ]:
cropdata = cropdata.dropna(subset=['Area','Production'])

In [ ]:
cropdata = cropdata[cropdata['Area']>0]

In [ ]:
cropdata = cropdata[cropdata['Production']>0]

In [ ]:
cropdata['Yield'] = cropdata['Production']/cropdata['Area']

In [ ]:
cropdata = cropdata.rename(columns={
    'Crop_Year': 'Year'
})
cropdata['State_Name']    = cropdata['State_Name'].str.strip().str.title()
cropdata['District_Name'] = cropdata['District_Name'].str.strip().str.title()
cropdata['Crop']          = cropdata['Crop'].str.strip().str.title()
cropdata['Season']        = cropdata['Season'].str.strip().str.title()

In [ ]:
print(cropdata['Season'].value_counts())
print("\n States: ", cropdata['State_Name'].nunique())
print("Crops: ",cropdata['Crop'].nunique())

In [ ]:
raindata = raindata[["SUBDIVISION","YEAR","ANNUAL"]]
raindata = raindata.rename(columns={
    'SUBDIVISION': 'State_Name',
    'YEAR': 'Year',
    'ANNUAL': 'Annual_Rainfall'
})
raindata['State_Name'] = raindata['State_Name'].str.strip().str.title()

In [ ]:
raindata['State_Name'].unique()

In [ ]:
subdivision_to_state = {
    'Andaman & Nicobar Islands': 'Andaman And Nicobar Islands',
    'Arunachal Pradesh': 'Arunachal Pradesh',
    'Assam & Meghalaya': 'Assam',
    'Bihar': 'Bihar',
    'Chattisgarh': 'Chhattisgarh',
    'Coastal Andhra Pradesh': 'Andhra Pradesh',
    'Coastal Karnataka': 'Karnataka',
    'East Madhya Pradesh': 'Madhya Pradesh',
    'East Rajasthan': 'Rajasthan',
    'East Uttar Pradesh': 'Uttar Pradesh',
    'Gangetic West Bengal': 'West Bengal',
    'Gujarat Region': 'Gujarat',
    'Haryana Delhi & Chandigarh': 'Haryana',
    'Himachal Pradesh': 'Himachal Pradesh',
    'Jammu & Kashmir': 'Jammu And Kashmir',
    'Jharkhand': 'Jharkhand',
    'Kerala': 'Kerala',
    'Konkan & Goa': 'Goa',
    'Lakshadweep': 'Lakshadweep',
    'Madhya Maharashtra': 'Maharashtra',
    'Marathwada': 'Maharashtra',
    'Naga Mani Mizo Tripura': 'Nagaland',
    'North Interior Karnataka': 'Karnataka',
    'Orissa': 'Odisha',
    'Punjab': 'Punjab',
    'Rayalaseema': 'Andhra Pradesh',
    'Saurashtra & Kutch': 'Gujarat',
    'South Interior Karnataka': 'Karnataka',
    'Sub Himalayan West Bengal & Sikkim': 'West Bengal',
    'Tamil Nadu': 'Tamil Nadu',
    'Telangana': 'Telangana',
    'Uttarakhand': 'Uttarakhand',
    'Vidarbha': 'Maharashtra',
    'West Madhya Pradesh': 'Madhya Pradesh',
    'West Rajasthan': 'Rajasthan',
    'West Uttar Pradesh': 'Uttar Pradesh',
}
raindata['State_Name'] = raindata['State_Name'].map(subdivision_to_state)

In [ ]:
raindata['State_Name'].unique()

In [ ]:
raindata = raindata.dropna(subset=['State_Name'])

In [ ]:
raindata = raindata.groupby(['State_Name', 'Year'])['Annual_Rainfall'].mean().reset_index()

In [ ]:
pricedata.head()

In [ ]:
pricedata = pricedata.rename(columns={
    'State':'State_Name',
    'Commodity':'Crop',
    'Modal Price':'Price'
})

In [ ]:
pricedata['State_Name'] = pricedata['State_Name'].str.strip().str.title()
pricedata['Crop']       = pricedata['Crop'].str.strip().str.title()
pricedata['Price']      = pd.to_numeric(pricedata['Price'], errors='coerce')

pricedata = pricedata.dropna(subset=['Price', 'Crop', 'State_Name'])
pricedata = pricedata[pricedata['Price'] > 0]

In [ ]:
price_agg = pricedata.groupby(['State_Name', 'Crop'])['Price'].mean().reset_index()
price_agg.columns = ['State_Name', 'Crop', 'Avg_Price']

print(price_agg.shape)
price_agg.head()


In [ ]:
merged = pd.merge(
    cropdata,
    raindata,
    on=['State_Name', 'Year'],
    how='left'
)
print("After rain merge:", merged.shape)
print("Rainfall nulls:", merged['Annual_Rainfall'].isnull().sum())

In [ ]:
merged = pd.merge(
    merged,
    price_agg,
    on=['State_Name', 'Crop'],
    how='left'
)
print("After price merge:", merged.shape)
print("Price nulls:", merged['Avg_Price'].isnull().sum())

In [ ]:
# merged = merged.dropna(subset=['Avg_Price'])

final_cols = ['State_Name', 'District_Name', 'Year', 'Season', 
              'Crop', 'Area', 'Production', 'Yield', 
              'Annual_Rainfall', 'Avg_Price']
merged = merged[final_cols]

print("Final shape:", merged.shape)
merged.head()

In [ ]:
matched = merged['Avg_Price'].notna().sum()
total = len(cropdata)
print(f"Price matched for {matched}/{total} rows ({100*matched/total:.1f}%)")

In [ ]:
unmatched_crops = merged[merged['Avg_Price'].isna()]['Crop'].unique()
matched_crops = price_agg['Crop'].unique()

print(f"Unmatched crop count: {len(unmatched_crops)}")
print("\nSample unmatched crops:")
print(sorted(unmatched_crops)[:30])

print("\nSample price dataset crops:")
print(sorted(matched_crops)[:30])

In [ ]:
from fuzzywuzzy import process

# Build a mapping: crop_df crop name → best matching price_agg crop name
price_crops = price_agg['Crop'].unique().tolist()

def fuzzy_match_crop(crop_name, choices, threshold=80):
    match, score = process.extractOne(crop_name, choices)
    return match if score >= threshold else None

# Build mapping dict for unmatched crops only
crop_name_mapping = {}
for crop in unmatched_crops:
    best_match = fuzzy_match_crop(crop, price_crops)
    if best_match:
        crop_name_mapping[crop] = best_match

print(f"Fuzzy mapped {len(crop_name_mapping)} crops")
print("\nSample mappings:")
for k, v in list(crop_name_mapping.items())[:15]:
    print(f"  '{k}' → '{v}'")

In [ ]:
manual_fixes = {
    'Moong(Green Gram)'      : 'Green Gram',        # fix wrong match
    'Sugarcane'              : 'Sugarcane',          # 'Sugar' is wrong
    'Other Cereals & Millets': None,                 # no good match, drop
    'Other Kharif Pulses'    : None,
    'Other Rabi Pulses'      : None,
    'Other Summer Pulses'    : None,
    'Other Vegetables'       : None,
    'Other Misc. Crops'      : None,
}

for crop, fix in manual_fixes.items():
    if fix is None:
        crop_name_mapping.pop(crop, None)
    else:
        crop_name_mapping[crop] = fix

In [ ]:
print("ALL 71 FUZZY MAPPINGS:")
print("-" * 50)
for k, v in sorted(crop_name_mapping.items()):
    print(f"  '{k}' → '{v}'")

In [ ]:
cropdata['Crop'] = cropdata['Crop'].replace(crop_name_mapping)
cropdata['Crop'] = cropdata['Crop'].fillna(cropdata['Crop'])

In [ ]:
# Apply fuzzy mapping to crop_df
cropdata['Crop'] = cropdata['Crop'].replace(crop_name_mapping)
# For already-matched crops, keep original name
cropdata['Crop'] = cropdata['Crop'].fillna(cropdata['Crop'])

# Re-merge from scratch using matched crop name
merged2 = pd.merge(cropdata, raindata, on=['State_Name', 'Year'], how='left')

merged2 = pd.merge(
    merged2,
    price_agg,
    on=['State_Name', 'Crop'],
    how='left'
)

merged2 = merged2.dropna(subset=['Avg_Price'])

# Check improvement
matched2 = merged2['Avg_Price'].notna().sum()
total = len(cropdata)
print(f"✅ After fuzzy fix: {matched2}/{total} rows ({100*matched2/total:.1f}%)")

In [ ]:
merged2.head()

In [ ]:
final_cols = ['State_Name', 'District_Name', 'Year', 'Season', 
              'Crop', 'Area', 'Production', 'Yield', 
              'Annual_Rainfall', 'Avg_Price']

merged2 = merged2[final_cols]

merged2.to_csv('../dataset/merged_crop_data.csv', index=False)
print("✅ Saved!")
print("Final shape:", merged2.shape)
merged2.head()